# A Missing Value Is Not a Zero

Article #30 companion: nine independent synthetic experiments, nine original figures, and eighteen CSV evidence tables. A missingness indicator R=1 means **observed** throughout. No field population or causal effect is estimated here.


## Missingness mechanisms are conditional independence claims

- MCAR: R is independent of observed and unobserved data.
- MAR: conditional on observed auxiliaries Z, R is independent of the missing value Y; this does not mean unconditional random missingness.
- MNAR: even after conditioning on the observed information available for identification, R may depend on unseen Y.

These conditions are relative to the data and model. The names alone do not identify a mechanism from an incomplete table.


In [ ]:
from pathlib import Path
import tempfile
from IPython.display import display, Image
from analysis import Design, EXPERIMENTS, FIGURES, SEED, run
print('Canonical seed:', SEED, '| experiments:', ', '.join(EXPERIMENTS))


## Recoverability and uncertainty

For Y observed when R=1 and fully observed Z, MAR means P(R=1|Y,Z)=P(R=1|Z). If its probability π(Z) is known and bounded away from zero, an inverse-probability mean can recover a marginal target. The study uses the self-normalized Hájek form, not a generic doubly robust estimator.

For normal outcomes with **known** variance, the dedicated MI function draws missing observations after drawing a posterior mean and pools within/between variation. This is not a general-purpose MICE implementation.


In [ ]:
import numpy as np
from analysis import normal_multiple_imputation, hajek_mean
rng = np.random.default_rng(123)
mu, variance = normal_multiple_imputation(rng, np.array([1.,2.,3.,4.]), 3)
assert np.isfinite(mu) and variance > 0
print('Illustrative MI mean and variance:', mu, variance)


## Short study execution without altering canonical evidence

The full article uses 24 independent outer repetitions in most experiments, and 300 in its interval-coverage experiment. Quick mode uses three repetitions in most studies, ten for coverage, and isolated output paths. It calls precisely the same synthetic generator, estimator, and figure functions.


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    summaries = run(Design(root=root, quick=True))
    tables = sorted((root/'results').glob('*_quick.csv'))
    figures = sorted((root/'figures').glob('*_quick.png'))
    assert len(tables) == 18 and len(figures) == 9
    print('Tables:', len(tables), '| figures:', len(figures))
    for study, figure in zip(EXPERIMENTS, FIGURES):
        print('\nStudy:', study, 'summary rows:', len(summaries[study]))
        display(Image(filename=str(root/'figures'/f'{figure}_quick.png')))


## Evidence boundaries

- Complete cases can have unbiased conditional regression coefficients while yielding biased marginal summaries.
- MAR corrections require usable auxiliaries and observation probabilities with adequate support. Oracle probabilities in a simulation are not observable in real data.
- MNAR pattern-mixture delta is a sensitivity assumption; the hidden complete-data truth is shown only because the synthetic generator supplies it.
- Proper MI is model-dependent and its uncertainty differs from pretending an imputed value was observed.
- A missingness indicator can be predictive only while the measurement workflow is stable.

See the manuscript for derivations, exact assumptions, limitations, source audit, and reporting protocol.
